# Notebook 02 — ColPali + CLIP Indexing

Builds two retrieval indexes over OpenI CXR images:
1. **ColPali v1.3** (primary) — late-interaction multimodal retrieval
2. **CLIP ViT-L/14** (baseline) — global embedding FAISS index

Both indexes are saved to Google Drive for persistence.

In [ ]:
import os, sys, importlib.util
from google.colab import drive, userdata

# ── Mount Drive ───────────────────────────────────────────────────────────
try:
    drive.mount('/content/drive')
except:
    drive.mount('/content/drive', force_remount=True)

# ── Clone or update repo ──────────────────────────────────────────────────
REPO_URL = 'https://github.com/mohamedtaha77/cxr-rag-system.git'
os.system(f'git clone -q {REPO_URL} /content/cxr-rag-system 2>/dev/null || git -C /content/cxr-rag-system pull -q')

# ── Install packages ─────────────────────────────────────────────────────
!pip install -q colpali-engine byaldi 'transformers>=4.45.0' accelerate open-clip-torch faiss-cpu
!pip install -q --upgrade torchao

print('✓ Packages installed')

In [ ]:
# ── Load retriever modules ────────────────────────────────────────────────
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

colpali_mod = load_module(
    "colpali_retriever",
    "/content/cxr-rag-system/src/retrieval/colpali_retriever.py"
)
clip_mod = load_module(
    "clip_retriever",
    "/content/cxr-rag-system/src/retrieval/clip_retriever.py"
)

ColPaliRetriever = colpali_mod.ColPaliRetriever
CLIPRetriever = clip_mod.CLIPRetriever

# ── Set paths ────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/cxr_rag'
IMAGES_DIR = '/content/openi/images'
COLPALI_INDEX_DIR = os.path.join(DRIVE_ROOT, 'colpali_index')
CLIP_INDEX_DIR = os.path.join(DRIVE_ROOT, 'clip_index')
HF_TOKEN = userdata.get('HF_TOKEN')

os.makedirs(COLPALI_INDEX_DIR, exist_ok=True)
os.makedirs(CLIP_INDEX_DIR, exist_ok=True)

print(f'✓ Paths configured')
print(f'  Images: {IMAGES_DIR}')
print(f'  ColPali index: {COLPALI_INDEX_DIR}')
print(f'  CLIP index: {CLIP_INDEX_DIR}')

## 1. Build ColPali Index

ColPali (late-interaction multimodal retrieval) indexes CXR images as visual documents. Downloads ~7 GB model once.

In [ ]:
import torch, gc

# Clear VRAM before loading ColPali
torch.cuda.empty_cache()
gc.collect()

print('Building ColPali index (~45 min)...')
retriever = ColPaliRetriever()
retriever.build_index(images_dir=IMAGES_DIR, index_save_dir=COLPALI_INDEX_DIR)
print('✓ ColPali index built and saved to Drive')

In [ ]:
# ── Test ColPali retrieval ───────────────────────────────────────────────
results = retriever.search('pleural effusion bilateral', k=3)
print(f'Top-{len(results)} ColPali results for: "pleural effusion bilateral"')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(results), figsize=(12, 4))
if len(results) == 1:
    axes = [axes]
for ax, r in zip(axes, results):
    if r['image']:
        ax.imshow(r['image'], cmap='gray')
    ax.set_title(f"Score: {r['score']:.3f}")
    ax.axis('off')
plt.suptitle('ColPali Top-3 Retrieved CXRs')
plt.tight_layout()
plt.show()

In [ ]:
# ── Free ColPali VRAM before CLIP ────────────────────────────────────────
del retriever
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Build CLIP Index

CLIP ViT-L/14 (global embedding baseline) indexes images for comparison with ColPali.

In [ ]:
import glob

image_paths = sorted([
    p for p in glob.glob(os.path.join(IMAGES_DIR, '**/*.png'), recursive=True)
])
print(f'Found {len(image_paths)} PNG images')

print('Building CLIP index (~30 min)...')
clip_retriever = CLIPRetriever()
clip_retriever.build_index(image_paths, batch_size=128)
clip_retriever.save_index(CLIP_INDEX_DIR)
print('✓ CLIP index built and saved to Drive')

In [ ]:
# ── Test CLIP retrieval ──────────────────────────────────────────────────
results_clip = clip_retriever.search_by_text('pleural effusion bilateral', k=3)
print(f'Top-{len(results_clip)} CLIP results for: "pleural effusion bilateral"')

fig, axes = plt.subplots(1, len(results_clip), figsize=(12, 4))
if len(results_clip) == 1:
    axes = [axes]
for ax, r in zip(axes, results_clip):
    if r['image']:
        ax.imshow(r['image'], cmap='gray')
    ax.set_title(f"Score: {r['score']:.3f}")
    ax.axis('off')
plt.suptitle('CLIP Top-3 Retrieved CXRs')
plt.tight_layout()
plt.show()

## Summary

Both indexes are now built and saved to Google Drive:
- **ColPali index**: `{COLPALI_INDEX_DIR}`
- **CLIP index**: `{CLIP_INDEX_DIR}`

Next: Run **Notebook 03** to build pipelines and compute evaluation metrics.